# Reversible Anonymization with LCEL and Fuzzy Matching

This notebook demonstrates:
1. Using PresidioReversibleAnonymizer with LangChain Expression Language (LCEL)
2. Implementing combined exact+fuzzy matching strategy
3. Full workflow from anonymization -> LLM processing -> deanonymization

In [ ]:
# Install required packages
%pip install --upgrade --quiet langchain langchain-experimental langchain-openai \
    presidio-analyzer presidio-anonymizer spacy Faker thefuzz

In [ ]:
from langchain_experimental.data_anonymizer import PresidioReversibleAnonymizer
from langchain_experimental.data_anonymizer.deanonymizer_matching_strategies import (
    combined_exact_fuzzy_matching_strategy,
)
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Initialize anonymizer with common PII types
anonymizer = PresidioReversibleAnonymizer(
    analyzed_fields=["PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "CREDIT_CARD"],
    faker_seed=42  # For deterministic fake data
)

In [ ]:
# Original sensitive text
text = """John Doe recently lost his wallet. 
Inside is $500 cash and his Visa card 4111 1111 1111 1111. 
Contact him at johndoe@example.com or 555-123-4567."""

# Anonymize the text
anonymized_text = anonymizer.anonymize(text)
print("Anonymized:\n", anonymized_text)

In [ ]:
# Create LCEL chain with anonymization and deanonymization
template = """Convert this message into a formal notification:
{anonymized_text}
"""

prompt = PromptTemplate.from_template(template)
llm = ChatOpenAI(temperature=0)

# Chain: anonymize -> LLM -> deanonymize
chain = (
    {"anonymized_text": anonymizer.anonymize} 
    | prompt 
    | llm 
    | (lambda msg: anonymizer.deanonymize(
        msg.content,
        deanonymizer_matching_strategy=combined_exact_fuzzy_matching_strategy
    ))
)

# Invoke the chain
response = chain.invoke(text)
print("\nFinal Response:\n", response)

In [ ]:
# Demonstrate fuzzy matching robustness
llm_altered_text = """
We regret to inform that Mr. Lynch (contactable at 734-413-1647 
or jamesmichael@example.com) mislaid a Visa card ending with 40262.
"""

# Deanonymize with different strategies
print("Without fuzzy matching:\n", anonymizer.deanonymize(llm_altered_text))
print("\nWith fuzzy matching:\n", anonymizer.deanonymize(
    llm_altered_text,
    deanonymizer_matching_strategy=combined_exact_fuzzy_matching_strategy
))

## Key Takeaways
- LCEL enables seamless integration of privacy-preserving steps
- Combined matching strategy handles LLM output variations
- Full reversibility maintains data utility while protecting PII
- Mapping persistence allows consistent anonymization across sessions